In [36]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import re
import os
import math
from tqdm import tqdm
import string
from sentence_transformers import SentenceTransformer



Skipping import of cpp extensions due to incompatible torch version 2.9.1+cpu for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0112 10:19:09.291000 1256 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [37]:
SentenceTransformer

sentence_transformers.SentenceTransformer.SentenceTransformer

In [13]:
# path = r"../../DataPreprocessing/inputs/cleaned_courses (1).pkl"
path = "../../DataPreprocessing/outputs/chunks_notdropduplicates"

In [14]:
def load_and_concat_pickles(directory_path: str):
    """
    Reads all .pkl files in a directory and concatenates them into a single DataFrame.
    
    Parameters:
        directory_path (str): Path to directory containing pickle files.
        
    Returns:
        pd.DataFrame
    """
    
    dataframes = []

    for file in os.listdir(directory_path):
        if file.lower().endswith(".pkl"):
            full_path = os.path.join(directory_path, file)
            df = pd.read_pickle(full_path)
            dataframes.append(df)

    if not dataframes:
        raise ValueError("No pickle files found in the specified directory.")

    return pd.concat(dataframes, ignore_index=True)


In [15]:
df = load_and_concat_pickles(path)

In [16]:
df.head()

,ipeds_id,cat_type,start_yr,end_yr,page_num,col_num,type,annote_id,Department,Number,Title,Description,Prerequisites,Credits,Teacher,decade,quinquennium,DepartmentCleaned
0,190415,both,1998,1999,431,1,courses,701991,government,605,Comparative Methods,This Seminar Provides A Survey Of Different Me...,,4,"{'Honorific': '', 'Name': 'J. Pontusson', 'Deg...",1990,1995,government
1,166638,ug,2009,2010,290,0,courses,378402,soil and water conservation,121,Remote Sensing Digital Processing And Analysis,This Course Is Not Described In The Text,,3,{},2000,2005,soil and water conservation
2,166638,ug,2009,2010,290,0,courses,378402,soil and water conservation,128,Principles Of Animal Behavior,This Course Is Not Described In The Text,,3,{},2000,2005,soil and water conservation
3,166638,ug,2009,2010,290,0,courses,378402,soil and water conservation,127,Avian Physiology,This Course Is Not Described In The Text,,3,{},2000,2005,soil and water conservation
4,166638,ug,2009,2010,290,0,courses,378402,soil and water conservation,126,Introduction To The Diseases Of Wildlife,This Course Is Not Described In The Text,,3,{},2000,2005,soil and water conservation


# Approach to find repeated courses

1) Identify repeated courses based on unique id_catalog and id_department_number_course
2) Validation of step 1) 
    - 2.1) Focus on false positives 
        - ?
            - if have same catalog, same department, same number, same course title??
    - 2.2) Focus on false negatives
        - NOTE: check distances between titles to check courses that were considered different but might be the same course. do the same for departments
        - TODO: in the drop duplicates process we are considering exact matches. but something like this requires pairwise combinations
            - embedding and cosine similarity -> sensitive for short things
                - this allows us to find things very close to each other
                - check other information retrieval techniques to find similar documents
            - jaccard -> good to find typos?
            - levistin difference -> good to find typos?
            - for for errors beyond typos?


Notes: 
- NOTE: check distances between titles to check courses that were considered different but might be the same course. do the same for departments
    - TODO: in the drop duplicates process we are considering exact matches. but something like this requires pairwise combinations
        - embedding and cosine similarity -> sensitive for short things
            - this allows us to find things very close to each other
            - check other information retrieval techniques to find similar documents
        - jaccard -> good to find typos?
        - levistin difference -> good to find typos?
        - for for errors beyond typos?
- NOTE: course number seems to be relevant and with limited error in identifying repeated courses (there are courses repeated ~40 times where the only difference is in the course number, and that is correct when looing at the data https://ccapi.uvm.edu/cat/pdfs/204796/204796_both_2018_2019 page 44 of the pdf)
    - TODO: adjust remove duplicates to account for feature "number"
- NOTE: some courses (Fiat Lux Freshman Seminars) appear multiple times in teh same catalog in different departments, but they are liekly the smae course



# TBD: Functions to be removed (?), potentially implemented DataProcessing 

In [17]:
###in the DB there is already a id_catalog. for this NLP project purpose we created it here to correct for the end year error
df["id_catalog"] = (
    df["ipeds_id"].astype(str)
    + "_"
    + df["cat_type"].astype(str)
    + "_"
    + df["start_yr"].astype(str)
    + "_"
    + df["end_yr"].astype(str)
)

In [20]:

df["id_course"] = (
    df["Number"].astype(str).fillna("")
    + "_"
    + df["Title"].fillna("")
)

In [21]:
df["id_dep_code"] = (
    df["Department"].fillna("")
    + "_"
    + df["Number"].fillna("")
)

In [22]:
df["id_department_course"] = (
    df["Department"].fillna("")
    + "_"
    + df["Title"].fillna("")
)

In [23]:
df["id_department_course_number"] = (
    df["Department"].fillna("")
    + "_"
    + df["Title"].fillna("")
    + "_"
    + df["Number"].fillna("")
)

In [24]:
df["Description_len"] = df["Description"].apply(
    lambda x: len(x) if pd.notnull(x) else 0
)

In [ ]:
df['id_catalog_department_course_number'] = df['id_catalog'] + '_'+  df['id_department_course_number']

In [29]:
df['bool_course_repeated_wout_number'] = df.duplicated(["id_department_course", "id_catalog"], keep=False)

In [30]:
df['bool_course_repeated'] = df.duplicated(["id_department_course_number", "id_catalog"], keep=False)

In [31]:
df['bool_course_repeated_wout_number'].value_counts()

bool_course_repeated_wout_number
False    3765991
True      852945
Name: count, dtype: int64

In [32]:
df['bool_course_repeated'].value_counts()

bool_course_repeated
False    4477245
True      141691
Name: count, dtype: int64

In [33]:
#TBD: difference bool_course_repeated_wout_number bool_course_repeated

# Duplicates - Validation false negatives

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

In [126]:
df['id_catalog'].value_counts()

id_catalog
110662_ug_2020_2021      14774
110662_ug_2019_2020      14764
204796_both_2015_2016    14699
204796_both_2014_2015    14504
110662_ug_2018_2019      14398
                         ...  
200280_both_1886_1887        1
163286_both_1907_1908        1
200280_both_1900_1901        1
200280_both_1887_1888        1
110635_misc_1964_1965        1
Name: count, Length: 1668, dtype: int64

In [127]:
# not_repeated_courses = df[df['bool_course_repeated'] == False]['Title'].fillna('').tolist()
##TODO: adjust code to compare per year and per institution (id_catalog)
cond_catalog_example = df['id_catalog'] == '110662_ug_2020_2021'
df_not_repeated_courses = df[(df['bool_course_repeated'] == False) & 
                             (cond_catalog_example)].sort_values(by=["id_catalog", "id_department_course_number"]).head(10000)
##TODO: not title, but title + number?
not_repeated_courses = df_not_repeated_courses['Title'].fillna('').tolist() #TODO: REMOVE THIS IS JUST A SAMPLE TO SPEED UP THE CODE

In [200]:
df_repeated_courses = df[df['bool_course_repeated'] == True]

In [203]:
df['Description'].value_counts().head(1000)

Description
Variable Credit Course                                                                                                                                                                                                       5195
See Schedule Of Courses For Specific Titles                                                                                                                                                                                  4727
Su Grade Only                                                                                                                                                                                                                3130
Three Credits                                                                                                                                                                                                                2757
No Description Provided                                                             

In [204]:
aux_ = df[df['Description']=='Role Of Essential Elements For Plant Growth Including Classical And Modern Approaches To The Study Of Ion Availability And Transport']

In [ ]:
#NOTE: Did not find "Mineral Nutrition Of Plants" in 

In [207]:
aux_[aux_['id_catalog'] == '231174_ug_1975_1976']

,ipeds_id,cat_type,start_yr,end_yr,page_num,col_num,type,annote_id,Department,Number,...,DepartmentCleaned,id_catalog,id_course,id_dep_code,id_department_course,id_department_course_number,Description_len,bool_course_repeated,bool_course_repeated_wout_number,id_catalog_department_course_number
2927414,231174,ug,1975,1976,286,0,courses,167177,biology,205,...,biology,231174_ug_1975_1976,205_Mineral Nutrition Of Plants,biology_205,biology_Mineral Nutrition Of Plants,biology_Mineral Nutrition Of Plants_205,132,False,True,231174_ug_1975_1976biology_Mineral Nutrition O...
2927415,231174,ug,1975,1976,286,0,courses,167177,biology,285,...,biology,231174_ug_1975_1976,285_Mineral Nutrition Of Plants,biology_285,biology_Mineral Nutrition Of Plants,biology_Mineral Nutrition Of Plants_285,132,False,True,231174_ug_1975_1976biology_Mineral Nutrition O...


In [206]:
aux_['id_catalog'].value_counts()

id_catalog
231174_ug_1975_1976    2
231174_ug_1998_1999    2
231174_ug_1986_1987    1
231174_ug_1988_1989    1
231174_ug_2010_2011    1
                      ..
231174_gr_2008_2009    1
231174_ug_1987_1988    1
231174_gr_2007_2008    1
231174_gr_1998_2000    1
231174_ug_2006_2007    1
Name: count, Length: 63, dtype: int64

In [201]:
df_repeated_courses.sort_values(by=["id_catalog", "id_department_course_number"]).head(10)

,ipeds_id,cat_type,start_yr,end_yr,page_num,col_num,type,annote_id,Department,Number,...,DepartmentCleaned,id_catalog,id_course,id_dep_code,id_department_course,id_department_course_number,Description_len,bool_course_repeated,bool_course_repeated_wout_number,id_catalog_department_course_number
4200361,100663,gr,2004,2006,193,0,courses,411576,biology,799,...,biology,100663_gr_2004_2006,799_Doctoral Dissertation Research,biology_799,biology_Doctoral Dissertation Research,biology_Doctoral Dissertation Research_799,44,True,True,100663_gr_2004_2006biology_Doctoral Dissertati...
4200712,100663,gr,2004,2006,267,0,courses,411613,biology,799,...,biology,100663_gr_2004_2006,799_Doctoral Dissertation Research,biology_799,biology_Doctoral Dissertation Research,biology_Doctoral Dissertation Research_799,35,True,True,100663_gr_2004_2006biology_Doctoral Dissertati...
4200362,100663,gr,2004,2006,193,0,courses,411576,biology,798,...,biology,100663_gr_2004_2006,798_Doctoral Nondissertation Research,biology_798,biology_Doctoral Nondissertation Research,biology_Doctoral Nondissertation Research_798,8,True,True,100663_gr_2004_2006biology_Doctoral Nondissert...
4200710,100663,gr,2004,2006,267,0,courses,411613,biology,798,...,biology,100663_gr_2004_2006,798_Doctoral Nondissertation Research,biology_798,biology_Doctoral Nondissertation Research,biology_Doctoral Nondissertation Research_798,0,True,True,100663_gr_2004_2006biology_Doctoral Nondissert...
3677474,100663,gr,2004,2006,44,1,courses,411815,bst,620,...,bst,100663_gr_2004_2006,620_Applied Matrix Algebra,bst_620,bst_Applied Matrix Algebra,bst_Applied Matrix Algebra_620,179,True,True,100663_gr_2004_2006bst_Applied Matrix Algebra_620
3677548,100663,gr,2004,2006,44,1,courses,411839,bst,620,...,bst,100663_gr_2004_2006,620_Applied Matrix Algebra,bst_620,bst_Applied Matrix Algebra,bst_Applied Matrix Algebra_620,179,True,True,100663_gr_2004_2006bst_Applied Matrix Algebra_620
3677473,100663,gr,2004,2006,44,1,courses,411815,bst,619,...,bst,100663_gr_2004_2006,619_Data Collection And Management,bst_619,bst_Data Collection And Management,bst_Data Collection And Management_619,278,True,True,100663_gr_2004_2006bst_Data Collection And Man...
4200043,100663,gr,2004,2006,174,0,courses,411686,bst,619,...,bst,100663_gr_2004_2006,619_Data Collection And Management,bst_619,bst_Data Collection And Management,bst_Data Collection And Management_619,174,True,True,100663_gr_2004_2006bst_Data Collection And Man...
3677472,100663,gr,2004,2006,44,1,courses,411815,bst,617,...,bst,100663_gr_2004_2006,617_Design And Analysis Of Clinical Dental Res...,bst_617,bst_Design And Analysis Of Clinical Dental Res...,bst_Design And Analysis Of Clinical Dental Res...,178,True,True,100663_gr_2004_2006bst_Design And Analysis Of ...
3677547,100663,gr,2004,2006,44,1,courses,411839,bst,617,...,bst,100663_gr_2004_2006,617_Design And Analysis Of Clinical Dental Res...,bst_617,bst_Design And Analysis Of Clinical Dental Res...,bst_Design And Analysis Of Clinical Dental Res...,178,True,True,100663_gr_2004_2006bst_Design And Analysis Of ...


In [ ]:
#TODO: CHECK TITLE NULLS

In [128]:
model = SentenceTransformer("all-MiniLM-L6-v2")
X = model.encode(not_repeated_courses, show_progress_bar=True)


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

In [129]:
cos_non_repeated_courses = cosine_similarity(X, X)

In [130]:
rows, cols = np.triu_indices_from(cos_non_repeated_courses, k=1)

In [131]:
cos_non_repeated_courses_unique = cos_non_repeated_courses[rows, cols]
idxs_cos_non_repeated_courses_unique = np.array(list(zip(rows,cols)))

In [147]:
# diff_cos_non_repeated_courses_unique=cos_non_repeated_courses_unique[cos_non_repeated_courses_unique<1]


top_cos = np.argsort(cos_non_repeated_courses_unique)[::-1]

In [149]:
series_cos_non_repeated_courses_unique =pd.Series(cos_non_repeated_courses_unique)

In [177]:
idxs_cos_non_repeated_courses_unique[38416148]

array([5187, 6414])

In [163]:
aux_ = series_cos_non_repeated_courses_unique[series_cos_non_repeated_courses_unique<0.99999999].sort_values()

In [176]:
aux_.tail(10000)

38416148    0.951022
36816362    0.951022
36816364    0.951022
36816372    0.951022
2786914     0.951022
              ...   
1445287     1.000000
1445251     1.000000
1446237     1.000000
1445687     1.000000
1446091     1.000000
Length: 10000, dtype: float32

In [145]:
idxs_cos_non_repeated_courses_unique[top_cos[:10]]

array([[ 145, 6244],
       [ 145, 5841],
       [ 145, 6390],
       [ 145, 5405],
       [ 145, 5441],
       [ 145, 5774],
       [ 145, 5757],
       [1634, 7075],
       [1634, 4891],
       [ 460, 1261]])

In [ ]:
##TODO: courses with same title, but different department or number
##TODO: important to ensure quality of department and number (but reporting of course numbers, for instance, seems to change across universities and years))

In [187]:
cond_large_number = df['Number'].apply(len)>4

In [199]:
df['DepartmentCleaned'].value_counts().head(6000)

DepartmentCleaned
history                      138433
english                      112336
music                        109055
mathematics                  108131
education                     95917
                              ...  
human biology and society        32
ncl                              32
greeks                           32
africana                         32
child care                       32
Name: count, Length: 6000, dtype: int64

In [198]:
df['DepartmentCleaned'].shape[0]

4618936

In [191]:
df[cond_large_number & (df['id_catalog'] == '233921_gr_1975_1976')].sample(10)

,ipeds_id,cat_type,start_yr,end_yr,page_num,col_num,type,annote_id,Department,Number,...,DepartmentCleaned,id_catalog,id_course,id_dep_code,id_department_course,id_department_course_number,Description_len,bool_course_repeated,bool_course_repeated_wout_number,id_catalog_department_course_number
688476,233921,gr,1975,1976,42,0,courses,689669,aerospace engineering,"5131, 5132, 5133",...,aerospace engineering,233921_gr_1975_1976,"5131, 5132, 5133_Rocket Propulsion Systems","aerospace engineering_5131, 5132, 5133",aerospace engineering_Rocket Propulsion Systems,aerospace engineering_Rocket Propulsion System...,537,False,False,233921_gr_1975_1976aerospace engineering_Rocke...
682799,233921,gr,1975,1976,189,0,courses,688994,None,042 (5203),...,None,233921_gr_1975_1976,042 (5203)_Statistical Inference Ii,_042 (5203),_Statistical Inference Ii,_Statistical Inference Ii_042 (5203),156,False,False,233921_gr_1975_1976_Statistical Inference Ii_0...
3276294,233921,gr,1975,1976,95,0,courses,101706,avs,"5161, 5162, 5163",...,avs,233921_gr_1975_1976,"5161, 5162, 5163_S Bivillar","avs_5161, 5162, 5163",avs_S Bivillar,"avs_S Bivillar_5161, 5162, 5163",56,False,False,"233921_gr_1975_1976avs_S Bivillar_5161, 5162, ..."
678076,233921,gr,1975,1976,68,0,courses,690706,biology fun,"5251, 5252, 5253",...,biology fun,233921_gr_1975_1976,"5251, 5252, 5253_Biology Of Fun","biology fun_5251, 5252, 5253",biology fun_Biology Of Fun,"biology fun_Biology Of Fun_5251, 5252, 5253",141,False,False,233921_gr_1975_1976biology fun_Biology Of Fun_...
3401540,233921,gr,1975,1976,64,0,courses,101753,urban design,"5791, 5792 (5191, 5291)",...,urban design,233921_gr_1975_1976,"5791, 5792 (5191, 5291)_Resource Development S...","urban design_5791, 5792 (5191, 5291)",urban design_Resource Development Studio I I,urban design_Resource Development Studio I I_5...,203,False,False,233921_gr_1975_1976urban design_Resource Devel...
688475,233921,gr,1975,1976,42,0,courses,689669,aerospace engineering,"5141, 5142",...,aerospace engineering,233921_gr_1975_1976,"5141, 5142_Boundary Layer Theory Heat Transfer","aerospace engineering_5141, 5142",aerospace engineering_Boundary Layer Theory H...,aerospace engineering_Boundary Layer Theory H...,527,False,False,233921_gr_1975_1976aerospace engineering_Bound...
3401521,233921,gr,1975,1976,64,0,courses,101753,urban design,"5771, 5772, 5773 (5161, 5261, 5361)",...,urban design,233921_gr_1975_1976,"5771, 5772, 5773 (5161, 5261, 5361)_Planning P...","urban design_5771, 5772, 5773 (5161, 5261, 5361)",urban design_Planning Problems Studio I Ii Iii,urban design_Planning Problems Studio I Ii Iii...,256,False,False,233921_gr_1975_1976urban design_Planning Probl...
625871,233921,gr,1975,1976,167,0,courses,682247,minerals engineering,"5141, 5142",...,minerals engineering,233921_gr_1975_1976,"5141, 5142_Interpretations In Solid State Mbta...","minerals engineering_5141, 5142",minerals engineering_Interpretations In Solid ...,minerals engineering_Interpretations In Solid ...,93,False,False,233921_gr_1975_1976minerals engineering_Interp...
3700031,233921,gr,1975,1976,122,0,courses,685638,engineering science and mechanics,"131, 5332",...,engineering science and mechanics,233921_gr_1975_1976,"131, 5332_Wave Propagation In Solids","engineering science and mechanics_131, 5332",engineering science and mechanics_Wave Propaga...,engineering science and mechanics_Wave Propaga...,190,False,True,233921_gr_1975_1976engineering science and mec...
3275858,233921,gr,1975,1976,135,0,courses,101720,fisheries and wildlife sciences,"501, 5502, 5503",...,fisheries and wildlife sciences,233921_gr_1975_1976,"501, 5502, 5503_Techniques In Wildlife Management","fisheries and wildlife sciences_501, 5502, 5503",fisheries and wildlife sciences_Techniques In ...,fisheries and wildlife sciences_Techniques In ...,77,False,False,233921_gr_1975_1976fisheries and wildlife scie...


In [ ]:
#DOUBT: the same course when associated with multiple department needs to have the same number? Trying to find repeated courses that were misspled 
# (cannot look solely at name since courses with similar names can be different if with different numer (economics vs advanced economics, example))

In [179]:
df_not_repeated_courses[df_not_repeated_courses['Number'] == '89']

,ipeds_id,cat_type,start_yr,end_yr,page_num,col_num,type,annote_id,Department,Number,...,DepartmentCleaned,id_catalog,id_course,id_dep_code,id_department_course,id_department_course_number,Description_len,bool_course_repeated,bool_course_repeated_wout_number,id_catalog_department_course_number
3107249,110662,ug,2020,2021,227,1,courses,193252,None,89,...,None,110662_ug_2020_2021,89_Honors Seminars,_89,_Honors Seminars,_Honors Seminars_89,51,False,False,110662_ug_2020_2021_Honors Seminars_89
4036966,110662,ug,2020,2021,621,2,courses,193178,a,89,...,a,110662_ug_2020_2021,89_Honors Freshman Seminar In Fiat Lux,a_89,a_Honors Freshman Seminar In Fiat Lux,a_Honors Freshman Seminar In Fiat Lux_89,0,False,False,110662_ug_2020_2021a_Honors Freshman Seminar I...
3116103,110662,ug,2020,2021,171,1,courses,192902,american indian studies,89,...,american indian studies,110662_ug_2020_2021,89_Honors Seminars,american indian studies_89,american indian studies_Honors Seminars,american indian studies_Honors Seminars_89,225,False,False,110662_ug_2020_2021american indian studies_Hon...
4038053,110662,ug,2020,2021,176,2,courses,193001,anthropology,89,...,anthropology,110662_ug_2020_2021,89_Honors Seminars,anthropology_89,anthropology_Honors Seminars,anthropology_Honors Seminars_89,132,False,False,110662_ug_2020_2021anthropology_Honors Seminar...
3107024,110662,ug,2020,2021,620,0,courses,193243,arabic,89,...,arabic,110662_ug_2020_2021,89_Honors Seminars,arabic_89,arabic_Honors Seminars,arabic_Honors Seminars_89,95,False,False,110662_ug_2020_2021arabic_Honors Seminars_89
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3104026,110662,ug,2020,2021,405,2,courses,193390,m1cw,89,...,m1cw,110662_ug_2020_2021,89_Honors Seminars,m1cw_89,m1cw_Honors Seminars,m1cw_Honors Seminars_89,342,False,False,110662_ug_2020_2021m1cw_Honors Seminars_89
3119176,110662,ug,2020,2021,564,1,courses,192762,mathematics,89,...,mathematics,110662_ug_2020_2021,89_Honors Seminars,mathematics_89,mathematics_Honors Seminars,mathematics_Honors Seminars_89,184,False,False,110662_ug_2020_2021mathematics_Honors Seminars_89
3109625,110662,ug,2020,2021,747,1,courses,193692,mclas,89,...,mclas,110662_ug_2020_2021,89_Honors Seminars,mclas_89,mclas_Honors Seminars,mclas_Honors Seminars_89,95,False,False,110662_ug_2020_2021mclas_Honors Seminars_89
3114514,110662,ug,2020,2021,628,0,courses,192306,middle eastern studies,89,...,middle eastern studies,110662_ug_2020_2021,89_Honors Seminars,middle eastern studies_89,middle eastern studies_Honors Seminars,middle eastern studies_Honors Seminars_89,95,False,False,110662_ug_2020_2021middle eastern studies_Hono...


In [178]:
df_not_repeated_courses.iloc[[5187, 6414]][['Number','Title','id_catalog']]

,Number,Title,id_catalog
3107962,189,Advanced Honors Seminars,110662_ug_2020_2021
3117852,89,Honors Seminars,110662_ug_2020_2021


In [158]:
from Levenshtein import distance

data = df_not_repeated_courses.iloc[[146,6822]]

distance(data['Title'].tolist()[0], data['Title'].tolist()[1])

23

In [139]:
df_not_repeated_courses.iloc[[9681,9797]].drop_duplicates()

,ipeds_id,cat_type,start_yr,end_yr,page_num,col_num,type,annote_id,Department,Number,...,DepartmentCleaned,id_catalog,id_course,id_dep_code,id_department_course,id_department_course_number,Description_len,bool_course_repeated,bool_course_repeated_wout_number,id_catalog_department_course_number
3104475,110662,ug,2020,2021,500,1,courses,193574,"molecular, cell, and developmental biology",M176,...,"molecular, cell, and developmental biology",110662_ug_2020_2021,M176_Auditory Neuroscience Of Speech Perceptio...,"molecular, cell, and developmental biology_M176","molecular, cell, and developmental biology_Aud...","molecular, cell, and developmental biology_Aud...",93,False,False,"110662_ug_2020_2021molecular, cell, and develo..."
3109579,110662,ug,2020,2021,634,1,courses,193697,music,M176,...,music,110662_ug_2020_2021,M176_Auditory Neuroscience Of Speech Perceptio...,music_M176,music_Auditory Neuroscience Of Speech Percepti...,music_Auditory Neuroscience Of Speech Percepti...,116,False,False,110662_ug_2020_2021music_Auditory Neuroscience...
